In [ ]:
import json
import csv

input_file = "weather_data.json"
output_file = "weather_data_clean.csv"

with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

metadata = data.get("metadata", {})
timeseries = data.get("timeseries", {})

all_dates = set()
for variable, values in timeseries.items():
    all_dates.update(values.keys())

all_dates = sorted(all_dates)
variables = sorted(timeseries.keys())

with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["station_id", "station_name", "latitude", "longitude", "date", *variables])

    for date in all_dates:
        row = [metadata.get("id", ""), metadata.get("name", ""), metadata.get("location", {}).get("latitude", ""), metadata.get("location", {}).get("longitude", ""), date]
        for var in variables:
            row.append(timeseries.get(var, {}).get(date, ""))

        writer.writerow(row)

print(f"CSV written to {output_file}")

In [ ]:
import pandas as pd

df = pd.read_csv("weather_data_clean.csv")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df_filtered = df[df["date"].dt.year.isin([2020, 2021, 2022])]
df_filtered.to_csv("weather_data_clean.csv", index=False)

print(df_filtered.head())
print(f"Filtered rows: {len(df_filtered)}")

In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd()

if (cwd / "data" / "processed" / "trips_clean.csv").exists():
    project_root = cwd
elif (cwd.parent / "data" / "processed" / "trips_clean.csv").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Could not find data/processed/trips_clean.csv from current notebook location")

trips_file = project_root / "data" / "processed" / "trips_clean.csv"
weather_file = project_root / "pipeline" / "weather_data_clean.csv"
output_file = project_root / "data" / "processed" / "trips_clean_with_weather.csv"

trips_df = pd.read_csv(trips_file)
weather_df = pd.read_csv(weather_file, usecols=["date", "PRCP", "TMAX", "TMIN"])

trips_df["date"] = pd.to_datetime(trips_df["started_at"]).dt.date
weather_df["date"] = pd.to_datetime(weather_df["date"]).dt.date

merged_df = trips_df.merge(weather_df, on="date", how="left")
merged_df.to_csv(output_file, index=False)

print(f"Saved merged file to: {output_file}")

In [ ]:


cwd = Path.cwd()

if (cwd / "data" / "processed").exists():
    project_root = cwd
elif (cwd.parent / "data" / "processed").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Could not find data/processed folder from current notebook location")

old_file = project_root / "data" / "processed" / "trips_clean.csv"
new_file = project_root / "data" / "processed" / "trips_clean_with_weather.csv"

print("Current working dir:", cwd)
print("Old file exists:", old_file.exists(), old_file)
print("New file exists:", new_file.exists(), new_file)

if old_file.exists():
    old_file.unlink()
    print(f"Deleted: {old_file}")

if new_file.exists():
    new_file.rename(old_file)
    print(f"Renamed {new_file} to {old_file}")
else:
    print(f"File not found: {new_file}")